# Train the MLBB tactical networks (free, in Google Colab)

**Nothing to set.** Upload your CSV files, run every cell (*Runtime → Run all*), download one file: `mlbb_policies.zip`.
Put it in **Download/MLBB-Research** - the app unpacks it and loads the right network by itself.

What it makes, from your recorded play **and** the live matches you watched, together:
- **one network for all heroes** - it is also told each hero's type (role, melee / ranged, starts fights or protects,
  early- or late-game), so it plays a tank differently from a marksman, even heroes with few recordings;
- **one network per hero** you recorded enough of (1500+ moments) - those play best.

While you play, the app uses your hero's own network if there is one, else the all-heroes one. Moments of the 6 s before
a death count less (they led to dying). Re-train every few matches: each run **continues from your last networks** (kept in My Drive/MLBB-Research/policy_models and inside mlbb_policies.zip) and learns on all your moments, so nothing it learned is lost.

**Before you start:** app → Learn page → *Export my recorded play (CSV)* and *Export live matches (CSV)*
(files in Downloads/MLBB-Research). *Runtime → Change runtime type*: CPU is enough.

In [ ]:
# ---- Started by the app (Learn page > Train)? Then this runs on Kaggle, unattended. ----
# It makes the notebook behave as in Colab: the files the app uploaded appear in "My Drive/MLBB-Research",
# downloads go to Kaggle's output (the app fetches them), and nothing waits for typing.
# In Colab this cell does nothing.
import os, sys, glob, shutil, types, zipfile, importlib.machinery
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working")
def roboflow_key():       # Kaggle: the key from roboflow_key.txt (no typing); "" = none
    f = os.path.join("/content/drive/MyDrive/MLBB-Research", "roboflow_key.txt")
    return open(f).read().strip() if os.path.exists(f) else ""
if ON_KAGGLE:
    DRIVE = "/content/drive/MyDrive/MLBB-Research"
    os.makedirs(DRIVE, exist_ok=True)
    # Kaggle UNZIPS uploaded zips: a folder holding images/ + labels/ was a picture zip. Zip it again
    # under its own name (the notebook reads mlbb-*-dataset-*.zip) instead of copying every picture.
    # the uploads this run needs (the app writes their names here). One that is not attached yet
    # (Kaggle was still processing it) is waited for, then fetched directly with kagglehub.
    APP_SOURCES = []
    import time
    def _attached(ref):
        slug = ref.split("/")[-1]
        return any(slug in p.split("/") for p in glob.glob("/kaggle/input/**/", recursive=True))
    ROOTS = ["/kaggle/input"]
    for _ in range(12):
        missing = [r for r in APP_SOURCES if not _attached(r)]
        if not missing: break
        print("waiting for", missing); time.sleep(15)
    for ref in [r for r in APP_SOURCES if not _attached(r)]:
        try:
            import kagglehub
            ROOTS.append(kagglehub.dataset_download(ref)); print("fetched", ref, "directly")
        except Exception as e:
            print("could not fetch", ref, ":", e)
    zipped_roots = []
    for img_dir in sorted(set(os.path.dirname(p) for R in ROOTS for p in glob.glob(R + "/**/images/*", recursive=True))):
        root = os.path.dirname(os.path.dirname(img_dir))          # the folder above "<name>/images"
        if any(root == r or root.startswith(r + "/") for r in zipped_roots): continue
        parts = [x for x in img_dir.split("/") if x.startswith("mlbb-minimap-dataset") or x.startswith("mlbb-screen-dataset")]
        first = os.listdir(img_dir)[0] if os.listdir(img_dir) else ""
        kind = "minimap" if first.startswith("mm_") else "screen"
        name = parts[0] if parts else f"mlbb-{kind}-dataset-{len(zipped_roots)}"
        if not name.startswith("mlbb-"): name = "mlbb-" + name
        out = os.path.join(DRIVE, name + ".zip")
        with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
            for f in glob.glob(root + "/**/*", recursive=True):
                if os.path.isfile(f): z.write(f, os.path.relpath(f, root))
        zipped_roots.append(root)
        print("pictures from the app:", name + ".zip", len(os.listdir(img_dir)), "pictures")
    for f in [f for R in ROOTS for f in glob.glob(R + "/**/*", recursive=True)]:
        if not os.path.isfile(f) or any(f.startswith(r + "/") for r in zipped_roots): continue
        if "/gemma" in f.lower(): continue          # an attached model (Gemma), not a file from the app
        dst = os.path.join(DRIVE, os.path.basename(f))
        if os.path.exists(dst): continue
        # small files are copied (the notebook may overwrite them), big ones linked
        if os.path.getsize(f) < 50_000_000: shutil.copy(f, dst)
        else: os.symlink(f, dst)
    # the last tactical networks (Kaggle unzipped mlbb_policies.zip): back where the notebook looks for them,
    # so the training CONTINUES from them instead of starting over
    for f in glob.glob(os.path.join(DRIVE, "*.keras")):
        os.makedirs(os.path.join(DRIVE, "policy_models"), exist_ok=True)
        shutil.move(f, os.path.join(DRIVE, "policy_models", os.path.basename(f)))
    print("Files from the app:", sorted(os.listdir(DRIVE)))
    if not os.listdir(DRIVE):
        raise SystemExit("No files from the app arrived on Kaggle (the upload may still have been processing): "
                         "tap Train again in the app in a few minutes")
    KAGGLE_OUT = "/kaggle/working"
    def _download(p):
        shutil.copy(p, KAGGLE_OUT); print("result:", os.path.basename(p))
    # Libraries also reach for other parts of google.colab (e.g. the converter imports its "output"):
    # anything asked for exists and does nothing
    class _Anything:
        def __getattr__(self, n): return _Anything()
        def __call__(self, *a, **k): return _Anything()
        def __bool__(self): return False
        def __iter__(self): return iter(())
    class _ColabModule(types.ModuleType):
        def __getattr__(self, n):
            if n.startswith("__"): raise AttributeError(n)
            return _Anything()
    _colab = _ColabModule("google.colab"); _colab.__path__ = []
    _colab.__spec__ = importlib.machinery.ModuleSpec("google.colab", None, is_package=True)   # (libraries look it up)
    _colab.files = types.SimpleNamespace(download=_download, upload=lambda: {})
    _colab.drive = types.SimpleNamespace(mount=lambda *a, **k: None)
    _colab.userdata = types.SimpleNamespace(get=lambda k: os.environ.get(k))
    import google; google.colab = _colab
    sys.modules["google.colab"] = _colab


In [ ]:
# TensorFlow and pandas are already installed in Colab
import numpy as np, pandas as pd, tensorflow as tf, math, json, io
print("TensorFlow", tf.__version__)

## 1. Load your files
Easiest: put all your `mlbb-my-play-*.csv` and `mlbb-live-*.csv` files in **My Drive/MLBB-Research** (Google Drive app:
upload them there) - the cell below takes them from there by itself. Otherwise it asks you to pick them on this device.

In [ ]:
# Your files: taken from Google Drive (My Drive/MLBB-Research) when they are there - nothing to upload.
# If that folder has none, you pick them on this device instead. (Google asks once to allow Drive access.)
import glob, os
from google.colab import files
DRIVE_DIR = "/content/drive/MyDrive/MLBB-Research"
paths = []
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for pat in ["mlbb-my-play-*.csv", "mlbb-live-*.csv"]:
        paths += sorted(glob.glob(f"{DRIVE_DIR}/{pat}"))
    paths = sorted(set(paths))
except Exception as e:
    print("Google Drive not used:", e)
if paths:
    print(f"Using {len(paths)} CSV files from Google Drive (My Drive/MLBB-Research):", ", ".join(os.path.basename(p) for p in paths))
else:
    print("No CSV files in My Drive/MLBB-Research - pick them on this device:")
    up = files.upload()
    paths = [os.path.abspath(k) for k in up]
frames = [pd.read_csv(f) for f in paths if f.endswith(".csv")]
df = pd.concat(frames, ignore_index=True)     # my-play and live files mix fine
# several exports overlap (each holds everything recorded until then; older ones survive a reinstall):
# the same moment counts once
n0 = len(df)
df = df.drop_duplicates(subset=[k for k in ("ts", "hero", "features", "source") if k in df.columns]).reset_index(drop=True)
print(len(df), "rows from", len(frames), "files" + (f" ({n0 - len(df)} duplicates dropped)" if n0 > len(df) else ""))

## 2. Train everything
Takes a few minutes. For each network it prints how good it is on your most recent play (which it did not train on):
the joystick number with every move counted equally (40%+ = the app uses its joystick choices) and the buttons good
enough to be used. Parts that are not good enough yet are simply not used by the app.

In [ ]:
# Nothing to set: this trains ONE network for all your heroes (it is told each hero's type) and
# ONE network per hero you recorded enough of, from your play AND the live matches together,
# then packs them all into mlbb_policies.zip for the app.
# (Advanced, normally left alone:)
LIVE_WEIGHT = 1.0       # live-match moments count like yours
DIED_WEIGHT = 0.3       # moments of the 6 s before a death count less (they led to dying)
WATCHED_WEIGHT = 0      # heroes you watched while YOU were dead: left out
MIN_HERO_MOMENTS = 1500 # a hero needs this many moments for its own network (else the all-heroes one plays it)
EPOCHS = 40
TYPE_SIZE = 11          # hero-type numbers (role x6, ranged, starts fights, protects, late game, early game)

import zipfile, re, os, shutil
tf.get_logger().setLevel("ERROR")

def parse(col):
    out = []
    for v in df[col].fillna("").astype(str):
        out.append([float(x) for x in v.split(",")] if v.strip() else None)
    return out

f1 = parse("features")
f2 = parse("features2") if "features2" in df.columns else [None] * len(df)
n40 = sum(1 for v in f2 if v is not None and len(v) == 40)
n32 = sum(1 for v in f2 if v is not None and len(v) in (32, 40))
N = 40 if n40 >= 1500 else 32 if n32 >= 1500 else 17          # richest situation with enough rows
use2 = N > 17
df["source"] = df["source"].fillna("me") if "source" in df.columns else "me"
df["died"] = df["died"].fillna(0).astype(int) if "died" in df.columns else 0
has_type = "hero_type" in df.columns
htype = parse("hero_type") if has_type else [None] * len(df)
if not has_type:
    print("NOTE: these CSV files have no hero types (exported before app v36.9). The all-heroes network is made "
          "without them - export again from the app for the better one.")
print(f"{len(df)} rows; situation: {N} numbers; moments: {(df['source'] == 'me').sum()} yours, "
      f"{(df['source'] == 'live').sum()} from live matches, {(df['died'] == 1).sum()} before a death")

# ---- every usable moment once ----
X, XT, MOVE, CASTS, W, TS, HEROES = [], [], [], [], [], [], []
for i, row in df.iterrows():
    hero = str(row["hero"]).strip()
    watched = hero.startswith("~")
    if not hero or (watched and WATCHED_WEIGHT <= 0): continue
    feat = f2[i] if use2 else f1[i]
    if feat is None or len(feat) < N: continue
    mx, my = float(row["move_x"]), float(row["move_y"])
    move = 8 if math.hypot(mx, my) < 0.4 else int(round(math.atan2(my, mx) / (math.pi / 4))) % 8
    c = int(row["casts"])
    wgt = WATCHED_WEIGHT if watched else 1.0
    if row["source"] == "live": wgt *= LIVE_WEIGHT
    if int(row["died"]) == 1: wgt *= DIED_WEIGHT
    if wgt <= 0: continue
    t = htype[i] if htype[i] is not None and len(htype[i]) == TYPE_SIZE else [0.0] * TYPE_SIZE
    X.append(feat[:N]); XT.append(t); MOVE.append(move); CASTS.append([(c >> b) & 1 for b in range(6)])
    W.append(wgt); TS.append(int(row["ts"])); HEROES.append(hero)
X = np.array(X, np.float32); XT = np.array(XT, np.float32); MOVE = np.array(MOVE); CASTS = np.array(CASTS, np.float32)
W = np.array(W, np.float32); TS = np.array(TS); HEROES = np.array(HEROES)
order = np.argsort(TS, kind="stable")
X, XT, MOVE, CASTS, W, TS, HEROES = X[order], XT[order], MOVE[order], CASTS[order], W[order], TS[order], HEROES[order]
# a press counts for its neighbours too (within 0.45 s, same hero): "a little early / late" is not wrong
wide = CASTS.copy()
for d in (-1, 1):
    j = np.arange(len(TS)) + d; ok = (j >= 0) & (j < len(TS)); jj = np.clip(j, 0, len(TS) - 1)
    near = ok & (np.abs(TS[jj] - TS) <= 450) & (HEROES[jj] == HEROES)
    wide[near] = np.maximum(wide[near], CASTS[jj][near])
CASTS = wide
FIGHT = (X[:, 6] > 0.5) | (X[:, 7] > 0)
if N >= 32: FIGHT |= X[:, 20] > 0
own = np.array([not h.startswith("~") for h in HEROES])
counts = pd.Series(HEROES[own]).value_counts()
print("moments per hero:", ", ".join(f"{h} {c}" for h, c in counts.items()))
assert len(X) >= 300, "Too little recorded play: play more matches with screen capture on, then export again."

BTN = ["attack", "skill 1", "skill 2", "ultimate", "spell", "recall"]
KEYS = ["attack", "s1", "s2", "ult", "spell", "recall"]
APP = np.array([0.5, 0.6, 0.6, 0.6, 0.6, 0.85], np.float32)   # the app's own press lines

# CONTINUED TRAINING: each network is also saved as a trainable copy (policy_models/<name>.keras, in
# My Drive/MLBB-Research and inside mlbb_policies.zip). Next time it starts FROM that copy and learns
# on all your moments (old + new together, so it does not drift away from what it knew).
PREV_DIRS = ["/content/drive/MyDrive/MLBB-Research/policy_models", "/content/prev_models"]
os.makedirs("/content/policy_models", exist_ok=True)
try:   # or the trainable copies inside an earlier mlbb_policies.zip in Drive
    zp = "/content/drive/MyDrive/MLBB-Research/mlbb_policies.zip"
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as z:
            for nm in z.namelist():
                if nm.startswith("policy_models/") and nm.endswith(".keras"): z.extract(nm, "/content/prev_models_zip")
        PREV_DIRS.append("/content/prev_models_zip/policy_models")
except Exception as e:
    print("(earlier mlbb_policies.zip not read:", e, ")")

def previous(name, n_in):
    for d in PREV_DIRS:
        p = f"{d}/{name}.keras"
        if os.path.exists(p):
            try:
                m = tf.keras.models.load_model(p, compile=False)
                if m.input_shape[-1] == n_in: return m, p
                print(f"  ({name}: the earlier network has {m.input_shape[-1]} inputs, now {n_in}: starting fresh)")
            except Exception as e:
                print(f"  ({name}: earlier network not usable: {e})")
    return None, None

def train(sel, with_type, hero_label, name):
    """Trains one network on the moments in `sel`; returns (tflite bytes, info dict, one-line report)."""
    Xs = np.concatenate([X[sel], XT[sel]], 1) if with_type else X[sel]
    M, C, Ws, F = MOVE[sel], CASTS[sel], W[sel], FIGHT[sel]
    cut = int(len(Xs) * 0.85)                          # the most recent 15% checks the result
    n_in = Xs.shape[1]
    model, prev_path = previous(name, n_in)
    if model is not None:
        inp = model.input; move_out, cast_out = model.outputs
        lr = 3e-4                                      # gentler: refine what it knows
    else:
        inp = tf.keras.Input(shape=(n_in,), name="situation")
        h = tf.keras.layers.Dense(128, activation="relu")(inp)
        h = tf.keras.layers.Dropout(0.1)(h)
        h = tf.keras.layers.Dense(64, activation="relu")(h)
        move_out = tf.keras.layers.Dense(9, activation="softmax", name="move")(h)
        cast_out = tf.keras.layers.Dense(6, activation="sigmoid", name="casts")(h)
        model = tf.keras.Model(inp, [move_out, cast_out])
        lr = 1e-3
    # lopsided recordings: every move counts as much as the others, a press as much as the no-presses
    Ft = F[:cut]; cnt = np.bincount(M[:cut][Ft], minlength=9); present = cnt > 0
    cw = np.clip(np.where(present, Ft.sum() / (max(1, present.sum()) * np.maximum(cnt, 1)), 0.0), 0.0, 15.0)
    sw_move = Ws[:cut] * cw[M[:cut]] * Ft
    pos = C[:cut].sum(0)
    POS_W = tf.constant(np.clip((len(C[:cut]) - pos) / np.maximum(pos, 1), 1.0, 30.0), tf.float32)
    def weighted_bce(y, p):
        p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
        return -tf.reduce_mean(POS_W * y * tf.math.log(p) + (1 - y) * tf.math.log(1 - p), axis=-1)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=["sparse_categorical_crossentropy", weighted_bce])
    model.fit(Xs[:cut], [M[:cut], C[:cut]], sample_weight=[sw_move, Ws[:cut]],
              validation_data=(Xs[cut:], [M[cut:], C[cut:]]), epochs=EPOCHS, batch_size=128, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
    # how good, on recent play it did not train on
    pm_all, pc = model.predict(Xs[cut:], verbose=0)
    FV = F[cut:]; pm = pm_all[FV]; yv = M[cut:][FV]
    per = [((pm.argmax(1) == k) & (yv == k)).sum() / (yv == k).sum() for k in range(9) if (yv == k).sum() > 0]
    bal = float(np.mean(per)) if per else 0.0
    THR, QUALITY = [], {}
    for b in range(6):
        y = C[cut:, b] > 0
        if y.sum() < 5: THR.append(float(APP[b])); continue
        best_t, best_f = float(APP[b]), -1.0
        for t in np.arange(0.05, 0.96, 0.05):
            p = pc[:, b] > t; tp = (p & y).sum(); f = 2 * tp / max(1, p.sum() + y.sum())
            if f > best_f: best_f, best_t = f, float(t)
        THR.append(best_t); p = pc[:, b] > best_t
        QUALITY[KEYS[b]] = {"checked": int(y.sum()), "right": float((p & y).sum() / max(1, p.sum())), "found": float((p & y).sum() / y.sum())}
    # export with the press lines built in (so they match the app's)
    logit = lambda v: np.log(v / (1 - v))
    SHIFT = tf.constant(logit(APP) - logit(np.clip(np.array(THR, np.float32), 0.05, 0.95)), tf.float32)
    class Calibrate(tf.keras.layers.Layer):
        def call(self, p):
            p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
            return tf.sigmoid(tf.math.log(p / (1 - p)) + SHIFT)
    export = tf.keras.Model(inp, tf.keras.layers.Concatenate(name="policy")([move_out, Calibrate()(cast_out)]))
    import contextlib
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        model.save(f"/content/policy_models/{name}.keras")      # the trainable copy for next time
    tfl = tf.lite.TFLiteConverter.from_keras_model(export).convert()     # (its long printout hidden)
    seen = np.bincount(yv, minlength=9)
    fewest = int(seen[seen > 0].min()) if (seen > 0).any() else 0
    info = {"hero": hero_label, "hero_type_inputs": bool(with_type), "inputs": int(n_in), "outputs": 15, "examples": int(len(Xs)),
            "move_balanced": bal, "move_checked": int(len(yv)), "move_fewest": fewest, "buttons": QUALITY, "press_lines": THR,
            "live_weight": LIVE_WEIGHT, "died_weight": DIED_WEIGHT}
    joy = "used" if bal >= 0.40 and len(yv) >= 300 and fewest >= 15 else "not yet"
    btns = [k for k, q in QUALITY.items() if q["checked"] >= 20 and q["right"] >= 0.4]
    report = ("continued from the last network | " if prev_path else "new | ") + f"{len(Xs)} moments | joystick {bal:.0%} ({joy}) | buttons used: {', '.join(btns) or 'none yet'}"
    return tfl, info, report

def safe(name): return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_") or "hero"

out = {}
print("\nall heroes (with hero types) ...", end=" ", flush=True)
tfl, info, rep = train(own | (WATCHED_WEIGHT > 0), has_type, "", "all")
out["all"] = (tfl, info); print(rep)
for hero, n in counts.items():
    sel = own & (HEROES == hero)
    if n < MIN_HERO_MOMENTS or FIGHT[sel].sum() < 300:
        print(f"{hero}: {n} moments - the all-heroes network plays it (needs {MIN_HERO_MOMENTS}+ for its own)")
        continue
    print(f"{hero} ...", end=" ", flush=True)
    tfl, info, rep = train(sel, False, hero, safe(hero))
    out[safe(hero)] = (tfl, info); print(rep)

with zipfile.ZipFile("/content/mlbb_policies.zip", "w") as z:
    for k, (tfl, info) in out.items():
        z.writestr(f"policy_{k}.tflite", tfl)
        z.writestr(f"policy_info_{k}.json", json.dumps(info))
        if os.path.exists(f"/content/policy_models/{k}.keras"): z.write(f"/content/policy_models/{k}.keras", f"policy_models/{k}.keras")
try:   # keep the trainable copies in Drive for the next run
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copytree("/content/policy_models", "/content/drive/MyDrive/MLBB-Research/policy_models", dirs_exist_ok=True)
        print("Trainable copies saved for next time: My Drive/MLBB-Research/policy_models")
except Exception as e:
    print("(trainable copies not saved to Drive:", e, ") - they are inside mlbb_policies.zip too")
print(f"\nmlbb_policies.zip: {len(out)} networks ({', '.join(out)}) - put it in Download/MLBB-Research")


## The fight-choice network
A small network that picks ALL_IN / POKE / FARM / RETREAT / GROUP in fights (from your saved fight moments). It goes into the same `mlbb_policies.zip`; the app uses it on the Model page switch *Fight-choice network in fights*. Skipped when there are no `mlbb-fights-*.csv` files.

In [ ]:
# ---- THE FIGHT-CHOICE NETWORK (goes into the same mlbb_policies.zip) ----
# From every saved fight moment (mlbb-fights-*.csv: yours, the AI's, the live matches you watched)
# and how each went 4 s later, a small network learns which option fits the moment:
# ALL_IN / POKE / FARM / RETREAT / GROUP. The app asks it in fights (well under 1 ms) - advice that
# its Kotlin checks must allow. Its input: the fight summary turned into 26 numbers, EXACTLY like
# the app's FightNet.features() (keep the two in step). No fight CSVs = this part is skipped.
import glob, os, re, json, zipfile, numpy as np, pandas as pd
FIGHT_OPTIONS = ["ALL_IN", "POKE", "FARM", "RETREAT", "GROUP"]
ROLES = ["tank", "fighter", "assassin", "mage", "marksman", "support"]

def fight_features(s):
    f = np.zeros(26, dtype=np.float32); t = str(s).lower()
    m = re.search(r"\(([a-z]+), (ranged|melee)\)", t)
    if m:
        if m.group(1) in ROLES: f[ROLES.index(m.group(1))] = 1
        f[6] = 1.0 if m.group(2) == "ranged" else 0.0
    m = re.search(r"time (\d+):(\d+)", t)
    if m: f[7] = min(1.0, (float(m.group(1)) + float(m.group(2)) / 60) / 20); f[8] = 1
    m = re.search(r"hp (\d+)%, level (\d+)", t)
    if m: f[9] = float(m.group(1)) / 100; f[10] = float(m.group(2)) / 15
    m = re.search(r"ready: ([a-z0-9 ]+?)(,|\n|$)", t)
    if m:
        r = m.group(1).strip()
        if r != "none":
            parts = [p for p in r.split(" ") if p.strip()]
            f[11] = len(parts) / 3; f[12] = 1.0 if "ult" in parts else 0.0
    if re.search(r"spell [a-z ]+ ready", t): f[13] = 1
    m = re.search(r"enemies close: (\d+)", t)
    if m: f[14] = min(1.0, float(m.group(1)) / 5)
    m = re.search(r"enemies close: \d+ \(hp ([0-9% ]+)\)", t)
    hps = [float(x.rstrip("%")) for x in m.group(1).split(" ") if x.rstrip("%").replace(".", "", 1).isdigit()] if m else []
    f[15] = (min(hps) if hps else 100.0) / 100; f[16] = (float(np.mean(hps)) if hps else 100.0) / 100
    f[17] = 1.0 if "nearest in reach" in t else 0.0
    m = re.search(r"allies close: (\d+)", t)
    if m: f[18] = min(1.0, float(m.group(1)) / 4)
    m = re.search(r"minions: (\d+)", t)
    if m: f[19] = min(1.0, float(m.group(1)) / 6)
    f[20] = 1.0 if "under enemy tower: yes" in t else 0.0
    m = re.search(r"missing from map: (\d+)", t)
    if m: f[21] = min(1.0, float(m.group(1)) / 5)
    m = re.search(r"scores: attack ([0-9.]+), retreat ([0-9.]+), risk ([0-9.]+)", t)
    if m: f[22], f[23], f[24], f[25] = float(m.group(1)), float(m.group(2)), float(m.group(3)), 1
    return f

fpaths = sorted(set(glob.glob("/content/drive/MyDrive/MLBB-Research/mlbb-fights-*.csv") + glob.glob("/content/mlbb-fights-*.csv")))
if not fpaths:
    print("No mlbb-fights-*.csv: the fight-choice network is skipped this time.")
else:
    fd = pd.concat([pd.read_csv(p) for p in fpaths], ignore_index=True)
    fd = fd.drop_duplicates(subset=["ts", "option", "summary"]).sort_values("ts").reset_index(drop=True)
    fd["summary"] = fd["summary"].astype(str)
    # (older app versions labelled every unclear moment of yours near minions as FARM: left out)
    old_farm = (fd.by_you == 1) & (fd.option == "FARM") & ~fd.summary.str.contains("scores:")
    fd = fd[~old_farm].reset_index(drop=True)
    # the same labels as the small model's notebook: choices that went well (yours count double),
    # and after a bad outcome / a death the safer answer (GROUP with allies near, else RETREAT)
    def fight_label(row):
        o, r, mine, died = row.option, float(row.reward), int(row.by_you) == 1, int(row.died) == 1
        if o not in FIGHT_OPTIONS: return None, 0
        if r >= 0.3 and not died: return o, (2 if mine else 1)
        if mine and r > -0.5 and not died: return o, 1
        if (r <= -0.5 or died) and o in ("ALL_IN", "POKE", "FARM"):
            m = re.search(r"allies close: (\d+)", row.summary)
            return ("GROUP" if m and int(m.group(1)) > 0 else "RETREAT"), 1
        return None, 0
    X, Y, Wt, I = [], [], [], []
    for i, row in fd.iterrows():
        a, w = fight_label(row)
        if a is None: continue
        X.append(fight_features(row.summary)); Y.append(FIGHT_OPTIONS.index(a)); Wt.append(w); I.append(i)
    X, Y, Wt, I = np.array(X), np.array(Y), np.array(Wt, dtype=np.float32), np.array(I)
    print(f"{len(fd)} fight moments -> {len(X)} usable; answers:", {o: int((Y == k).sum()) for k, o in enumerate(FIGHT_OPTIONS)})
    if len(X) < 80:
        print("Too few usable fight moments for the fight-choice network (needs 80+): play / watch more fights.")
    else:
        cut = int(len(X) * 0.85)                      # the newest 15% are the test (fights it never saw)
        Xtr, Ytr, Wtr, Xte, Yte = X[:cut], Y[:cut], Wt[:cut], X[cut:], Y[cut:]
        # every answer counts the same in training (FARM must not drown the others)
        counts = np.bincount(Ytr, minlength=5).astype(np.float32)
        cw = np.where(counts > 0, counts.sum() / (5 * np.maximum(counts, 1)), 0.0)
        sw = Wtr * cw[Ytr]
        inp = tf.keras.Input(shape=(26,), name="fight")
        h = tf.keras.layers.Dense(64, activation="relu")(inp)
        h = tf.keras.layers.Dropout(0.2)(h)
        h = tf.keras.layers.Dense(32, activation="relu")(h)
        out = tf.keras.layers.Dense(5, activation="softmax", name="option")(h)
        fnet = tf.keras.Model(inp, out)
        fnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy")
        fnet.fit(Xtr, Ytr, sample_weight=sw, epochs=80, batch_size=32, verbose=0,
                 validation_data=(Xte, Yte) if len(Xte) else None,
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)] if len(Xte) else [])
        acc = base = bal = 0.0
        if len(Xte):
            pred = fnet.predict(Xte, verbose=0).argmax(1)
            acc = float((pred == Yte).mean()); base = float((Yte == np.bincount(Ytr).argmax()).mean())
            per = [float((pred[Yte == k] == k).mean()) for k in range(5) if (Yte == k).any()]
            bal = float(np.mean(per)) if per else 0.0
            print(f"fight-choice network: right {acc:.0%} (always the most common answer: {base:.0%}), "
                  f"balanced {bal:.0%} (each answer equally; chance = 20%)")
            print("answers it gave:", {FIGHT_OPTIONS[k]: int((pred == k).sum()) for k in range(5)})
        conv = tf.lite.TFLiteConverter.from_keras_model(fnet)
        open("/content/fight_choice.tflite", "wb").write(conv.convert())
        finfo = {"features": 26, "classes": FIGHT_OPTIONS, "acc": acc, "base": base, "balanced": bal,
                 "test": int(len(Xte)), "train": int(len(Xtr))}
        with zipfile.ZipFile("/content/mlbb_policies.zip", "a") as z:
            z.write("/content/fight_choice.tflite", "fight_choice.tflite")
            z.writestr("fight_choice_info.json", json.dumps(finfo))
        print("fight_choice.tflite added to mlbb_policies.zip")


## 3. Download

In [ ]:
import shutil
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/mlbb_policies.zip")
try:   # also keep a copy in My Drive/MLBB-Research when Google Drive is connected
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
        shutil.copy("/content/mlbb_policies.zip", "/content/drive/MyDrive/MLBB-Research/")
        print("Also saved to Google Drive > My Drive/MLBB-Research")
except Exception as e: print("(not copied to Drive:", e, ")")

In [ ]:
# B. Save to Google Drive (most reliable on a phone)
from google.colab import drive; import shutil, os
drive.mount("/content/drive"); os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
shutil.copy("/content/mlbb_policies.zip", "/content/drive/MyDrive/MLBB-Research/mlbb_policies.zip")
print("Saved to Google Drive > My Drive/MLBB-Research: mlbb_policies.zip")

## 4. Put it in the app
1. Put `mlbb_policies.zip` in **Download/MLBB-Research** on the phone (replace the old one). The app unpacks it by itself
   (Model page: *Scan the folder now*, or restart the app).
2. The Model page lists the networks on the phone; while playing, the right one is loaded when your hero is known.
3. Only the parts each network got good at are used; the safety rules (retreat, tower circle, recall) always come first.

In [ ]:
# ---- Kaggle: every result the app downloads, in Kaggle's output ----
if ON_KAGGLE:
    for name in ['mlbb_policies.zip']:
        for src in (f"/content/{name}", f"{DRIVE}/{name}"):
            if os.path.isfile(src):
                shutil.copy(src, KAGGLE_OUT); break
    print("Results for the app:", sorted(os.listdir(KAGGLE_OUT)))
